# Phase 05 — Data Formatting & Prompt Template Design
## CodeMentor-LLM
Converting CodeAlpaca-20K dataset to Llama-3 chat template format
for supervised fine-tuning (SFT).

## Goal
- Study Llama-3 chat template
- Write format_prompt() function
- Apply to entire dataset
- Save formatted dataset to HuggingFace Hub

# Libraries

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pandas==2.2.3

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

# Load primary dataset from HF Hub
print("Loading primary dataset...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k")
print(dataset)
print(f"\nTotal samples: {len(dataset['train'])}")

# Study Llama-3 chat template

In [ ]:
from transformers import AutoTokenizer

# Load Llama-3 tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Study the chat template with a sample message
messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to reverse a string."},
    {"role": "assistant", "content": "Here is a Python function to reverse a string:\n```python\ndef reverse_string(s):\n    return s[::-1]\n```"}
]

# Apply chat template
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Llama-3 Chat Template Format:")
print("=" * 50)
print(formatted)

# Write format_prompt() function

In [ ]:
SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def format_prompt(sample):
    """
    Converts a CodeAlpaca sample to Llama-3 chat template format.
    Args:
        sample: dict with keys instruction, input, output
    Returns:
        dict with key text containing formatted prompt
    """
    # Combine instruction and input if input exists
    if sample["input"] and sample["input"].strip():
        user_message = f"{sample['instruction']}\n\nInput:\n{sample['input']}"
    else:
        user_message = sample["instruction"]

    # Build messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": sample["output"]}
    ]

    # Apply Llama-3 chat template
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted}

print("format_prompt() function defined successfully")

# Test format_prompt() on single sample

In [ ]:
# Test on first sample
sample = dataset["train"][0]
print("ORIGINAL SAMPLE:")
print(f"Instruction: {sample['instruction']}")
print(f"Input: {sample['input']}")
print(f"Output: {sample['output']}")

print("\n" + "="*50)
print("FORMATTED SAMPLE:")
print("="*50)
formatted_sample = format_prompt(sample)
print(formatted_sample["text"])

# Test with sample that has input field

In [ ]:
# Find a sample that has input field
for i, sample in enumerate(dataset["train"]):
    if sample["input"] and sample["input"].strip():
        print(f"Sample index: {i}")
        print(f"Instruction: {sample['instruction']}")
        print(f"Input: {sample['input']}")
        print(f"Output: {sample['output']}")
        print("\n" + "="*50)
        print("FORMATTED:")
        print("="*50)
        formatted = format_prompt(sample)
        print(formatted["text"])
        break

# Apply format_prompt() to entire dataset

In [ ]:
# Apply to entire dataset
print("Formatting entire dataset...")
formatted_dataset = dataset["train"].map(
    format_prompt,
    remove_columns=dataset["train"].column_names
)

print(f"Formatted dataset size: {len(formatted_dataset)}")
print(f"Columns: {formatted_dataset.column_names}")
print(f"\nFirst formatted sample:")
print(formatted_dataset[0]["text"][:200])

# Check token lengths

In [ ]:
# Check token lengths of formatted samples
print("Checking token lengths...")

token_lengths = []
for sample in formatted_dataset:
    tokens = tokenizer(sample["text"], return_tensors="pt")
    token_lengths.append(tokens["input_ids"].shape[1])

import pandas as pd
lengths_series = pd.Series(token_lengths)

print(f"Min tokens  : {lengths_series.min()}")
print(f"Max tokens  : {lengths_series.max()}")
print(f"Mean tokens : {lengths_series.mean():.2f}")
print(f"Median      : {lengths_series.median()}")
print(f"\nSamples under 2048 tokens : {(lengths_series <= 2048).sum()}")
print(f"Samples over 2048 tokens  : {(lengths_series > 2048).sum()}")

# Push formatted dataset to HuggingFace Hub

In [ ]:
from datasets import Dataset

# Convert to Dataset object
formatted_dataset_hf = Dataset.from_dict({"text": formatted_dataset["text"]})

# Push to HF Hub
print("Pushing formatted dataset to HF Hub...")
formatted_dataset_hf.push_to_hub("Abdulmoiz123/codementor-llm-formatted")
print("Formatted dataset pushed successfully")
print(f"Total samples: {len(formatted_dataset_hf)}")

# Formatted data

In [ ]:
# Save formatted dataset as JSONL for download
import json

# Save formatted dataset
with open("formatted_dataset.jsonl", "w") as f:
    for sample in formatted_dataset_hf:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(formatted_dataset_hf)} samples to formatted_dataset.jsonl")

# Raw data

In [ ]:
import json

# Save primary raw dataset
with open("codealapaca_20k_raw.jsonl", "w") as f:
    for sample in dataset["train"]:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(dataset['train'])} samples to codealapaca_20k_raw.jsonl")

# Backup data

In [ ]:
# Save backup raw dataset
from datasets import load_dataset

backup_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca")

with open("python_code_instructions_18k_raw.jsonl", "w") as f:
    for sample in backup_dataset["train"]:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(backup_dataset['train'])} samples to python_code_instructions_18k_raw.jsonl")